# Dajjal Part 2: render final (wayang 2D, Three.js)
Upload `dajjal-part2.zip` terbaru ke **My Drive** (folder paling atas), runtime **G4**, lalu **Runtime → Run all**.

In [ ]:
ZIP = '/content/drive/MyDrive/dajjal-part2.zip'
OUT_NAME = 'Simulasi.WNI_Dajjal_Part2.mp4'
PART = 2
WORKERS = 3
!nvidia-smi -L || true

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/dajjal-part2 /content/chunks && unzip -q -o "$ZIP" -d /content
!ls /content/dajjal-part2

In [ ]:
!pip -q install playwright
!playwright install --with-deps chromium > /tmp/pw.log 2>&1; tail -2 /tmp/pw.log
!apt-get -qq remove -y mesa-vulkan-drivers > /dev/null 2>&1 || true
!apt-get -qq install -y ffmpeg libvulkan1 libegl1 libgles2 > /dev/null 2>&1; echo apt ok

In [ ]:
# Sambungkan driver NVIDIA ke Vulkan + EGL
import glob, os, json
libdirs = [d for d in ['/usr/lib64-nvidia','/usr/lib/x86_64-linux-gnu','/usr/local/nvidia/lib64'] if os.path.isdir(d)]
def find(n):
    for d in libdirs:
        h = glob.glob(f'{d}/{n}*')
        if h: return sorted(h)[0]
glx, egl = find('libGLX_nvidia.so'), find('libEGL_nvidia.so')
print('libGLX_nvidia:', glx, '| libEGL_nvidia:', egl)
assert glx, 'Driver NVIDIA tidak ketemu. Pastikan runtime GPU (G4).'
os.makedirs('/usr/share/vulkan/icd.d', exist_ok=True); os.makedirs('/usr/share/glvnd/egl_vendor.d', exist_ok=True)
ICD = '/usr/share/vulkan/icd.d/nvidia_icd.json'
json.dump({'file_format_version':'1.0.0','ICD':{'library_path':glx,'api_version':'1.3.0'}}, open(ICD,'w'))
if egl: json.dump({'file_format_version':'1.0.0','ICD':{'library_path':egl}}, open('/usr/share/glvnd/egl_vendor.d/10_nvidia.json','w'))
os.environ['VK_ICD_FILENAMES'] = ICD; os.environ['VK_DRIVER_FILES'] = ICD
os.environ['LD_LIBRARY_PATH'] = ':'.join(libdirs) + ':' + os.environ.get('LD_LIBRARY_PATH','')

In [ ]:
# Pilih renderer: NVIDIA lewat Vulkan, kalau gagal pakai CPU
import subprocess, re, time
os.chdir('/content/dajjal-part2')
GPU = {'GL_ARGS': json.dumps(['--use-angle=vulkan','--enable-features=Vulkan,VulkanFromANGLE,DefaultANGLEVulkan','--enable-gpu','--disable-vulkan-surface']), 'HEADLESS_NEW': '1'}
CPU = {'GL_ARGS': json.dumps(['--use-angle=swiftshader','--enable-unsafe-swiftshader'])}
CHOICE = None
for name, env in [('GPU vulkan', GPU), ('CPU swiftshader', CPU)]:
    r = subprocess.run(['python3','render.py','probe',str(PART),'45'], env=dict(os.environ, **env), capture_output=True, text=True, timeout=1200)
    o = r.stdout + r.stderr
    rend = re.findall(r'RENDERER (.*)', o); spf = re.findall(r'detik/frame ([\d.]+)', o)
    print(f'{name}: renderer={rend[:1]} detik/frame={spf[:1]} ok={r.returncode==0}')
    if r.returncode == 0 and rend and spf:
        if name.startswith('GPU') and any(x in rend[0] for x in ['SwiftShader','llvmpipe','lavapipe']): continue
        CHOICE = (name, env); break
    else: print(o[-1500:])
print('DIPAKAI:', CHOICE[0])

In [ ]:
# Render paralel: tiap worker membuat satu potongan mp4 (potongan yang sudah ada dilewati)
import math
name, env = CHOICE
DUR = json.load(open(f'web/data_part{PART}.json'))['duration']
FRAMES = round(DUR * 30)
os.makedirs('/content/chunks', exist_ok=True)
chunk = math.ceil(FRAMES / WORKERS); procs = []
for i in range(WORKERS):
    lo, hi = i * chunk, min(FRAMES - 1, (i + 1) * chunk - 1)
    log = open(f'/content/w{i}.log', 'w')
    procs.append(subprocess.Popen(['python3','render.py','chunk',str(PART),str(lo),str(hi),f'/content/chunks/c{i:02d}.mp4'], env=dict(os.environ, **env), stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(20)
    done = []
    for i in range(WORKERS):
        m = re.findall(r'frame (\d+)', open(f'/content/w{i}.log').read())
        done.append(int(m[-1]) - i * chunk if m else 0)
    n = sum(done); rate = n / max(1, time.time() - t0)
    print(f'{n}/{FRAMES} frame, sisa ~{(FRAMES-n)/rate/60 if rate>0 else float("nan"):.1f} menit', flush=True)
codes = [p.returncode for p in procs]; print('selesai:', codes)
for i, c in enumerate(codes):
    if c: print(f'--- log worker {i} ---'); print(open(f'/content/w{i}.log').read()[-2000:])

In [ ]:
# Gabung potongan + VO jadi MP4
import shutil
parts = [f'/content/chunks/c{i:02d}.mp4' for i in range(WORKERS)]
miss = [p for p in parts if not os.path.exists(p)]
assert not miss, f'Potongan hilang: {miss}. Jalankan ulang sel render.'
open('/content/chunks/list.txt','w').write(''.join(f"file '{p}'\n" for p in parts))
!ffmpeg -v error -y -f concat -safe 0 -i /content/chunks/list.txt -c copy /content/silent.mp4
!ffmpeg -v error -y -i /content/silent.mp4 -i /content/dajjal-part2/vo_part{PART}.wav -map 0:v -map 1:a -c:v copy -c:a aac -b:a 192k -shortest -movflags +faststart /content/out.mp4
shutil.copy('/content/out.mp4', '/content/drive/MyDrive/' + OUT_NAME)
print('Tersimpan di My Drive:', OUT_NAME)
from google.colab import files
files.download('/content/out.mp4')